# RetinAI CNN v2 — multitarea con etiquetas enmascaradas

Modelo clínico académico con tres detectores independientes: retinopatía diabética, glaucoma y catarata. La probabilidad normal se deriva como `1 - máximo(enfermedades)`, por lo que la salida final conserva los cuatro porcentajes exigidos por RetinAI.

Usa las cinco fuentes acordadas: ODIR, Sylhet depurado, cuatro clases sin copias ODIR, IDRiD y RIM-ONE DL. Las etiquetas ausentes en IDRiD/RIM-ONE se ignoran en vez de convertirse falsamente en diagnósticos negativos.

Durante cada época verás en pantalla accuracy, AUC, F1, sensibilidad y especificidad de validación para cada enfermedad.

In [ ]:
%pip -q install kagglehub gdown pandas scikit-learn importlib-resources
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU:', tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    raise RuntimeError('Selecciona GPU T4 en Entorno de ejecución > Cambiar tipo de entorno.')


In [ ]:
from google.colab import drive
from pathlib import Path
drive.mount('/content/drive')
destino = Path('/content/drive/MyDrive/RETINAI_CNN_V2')
requeridos = [
    destino / 'entrenar_clasificador_multifuente_real_v2.py',
    destino / 'entrenar_multitarea_enmascarado_v2.py',
    destino / 'retinai_fuentes_v2.csv',
    destino / 'odir_manifest_v2_split.csv',
    destino / 'sylhet_manifest_v2.csv',
]
faltan = [str(ruta) for ruta in requeridos if not ruta.is_file()]
if faltan:
    raise RuntimeError(f'Faltan archivos auxiliares: {faltan}')
print('Archivos y manifiestos verificados.')


## Entrenamiento con calidad visible

Esta celda muestra en vivo el progreso. Después de cada época aparecerá `CALIDAD VALIDACION` con las métricas de diabetes, glaucoma y catarata.

In [ ]:
import os, runpy, sys
os.environ['PYTHONUNBUFFERED'] = '1'
script = destino / 'entrenar_multitarea_enmascarado_v2.py'
sys.path.insert(0, str(destino))
sys.argv = [str(script), str(destino)]
runpy.run_path(str(script), run_name='__main__')


## Resumen final en pantalla

In [ ]:
import json, pandas as pd
from IPython.display import display
informe = destino / 'informe_clasificador_multitarea_enmascarado_v2.json'
datos = json.loads(informe.read_text(encoding='utf-8'))
print('DECISIÓN:', datos['decision'])
print('TAMAÑO DE ENTRADA:', datos['image_size'], 'x', datos['image_size'])
print('UMBRALES:', json.dumps(datos['thresholds'], ensure_ascii=False, indent=2))
filas = []
for fuente, evaluacion in datos['evaluations'].items():
    if 'classification' in evaluacion:
        filas.append({'fuente': fuente, 'tarea': '4 clases', 'accuracy': evaluacion['classification']['accuracy'], 'f1': evaluacion['classification']['macro_f1'], 'auc': None, 'sensibilidad': None, 'especificidad': None})
        enfermedades = evaluacion['diseases']
    else:
        enfermedades = {fuente.split('_')[-1]: evaluacion}
    for enfermedad, metricas in enfermedades.items():
        filas.append({'fuente': fuente, 'tarea': enfermedad, 'accuracy': metricas['accuracy'], 'f1': metricas['f1'], 'auc': metricas['auc'], 'sensibilidad': metricas['sensitivity'], 'especificidad': metricas['specificity']})
display(pd.DataFrame(filas).round(4))
print('CRITERIOS:')
display(pd.DataFrame([{'criterio': clave, 'cumple': valor} for clave, valor in datos['acceptance_criteria'].items()]))
print('Keras candidato:', datos['keras_candidate'])
print('TFLite:', datos['tflite_exported'] or 'NO GENERADO')
from google.colab import files
files.download(str(informe))


## Prueba manual: sube imágenes de cualquier dataset

Puedes seleccionar varias imágenes. La celda mostrará la imagen y los cuatro porcentajes que recibirá el sistema.

In [ ]:
from google.colab import files
from PIL import Image, ImageOps
import io, numpy as np, matplotlib.pyplot as plt
import entrenar_multitarea_enmascarado_v2  # registra la pérdida personalizada
modelo = tf.keras.models.load_model(destino / 'clasificador_multitarea_enmascarado_v2_candidato.keras')
subidas = files.upload()
for nombre, contenido in subidas.items():
    imagen = Image.open(io.BytesIO(contenido)).convert('RGB')
    preparada = ImageOps.pad(imagen, (384, 384), method=Image.Resampling.BILINEAR, color=(0, 0, 0))
    lote = np.expand_dims(np.asarray(preparada, dtype=np.float32), 0)
    enfermedades = modelo.predict(lote, verbose=0)[0]
    porcentajes = np.concatenate(([1.0 - enfermedades.max()], enfermedades)) * 100.0
    etiquetas = ['Normal', 'Retinopatía diabética', 'Glaucoma', 'Catarata']
    fig, ejes = plt.subplots(1, 2, figsize=(12, 4))
    ejes[0].imshow(imagen); ejes[0].set_title(nombre); ejes[0].axis('off')
    ejes[1].barh(etiquetas, porcentajes, color=['#16a34a', '#dc2626', '#ea580c', '#2563eb'])
    ejes[1].set_xlim(0, 100); ejes[1].set_xlabel('Porcentaje')
    for indice, valor in enumerate(porcentajes): ejes[1].text(min(valor + 1, 96), indice, f'{valor:.1f}%', va='center')
    plt.tight_layout(); plt.show()
